# Figure 1c — response genes vs IRIS (in-sample)

The starting question of the paper: are the handful of empirically annotated
response genes (*Axin2* for WNT, *Id1* for BMP, …) enough to call pathway
activity across cell types?

Here both methods are scored on the same held-out 20% of the mESC screen, so
the comparison is like-for-like.

Script equivalent: `figures/fig1/fig1c_insample_accuracy.py`

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src" / "iris_repro").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "figures"))

import numpy as np
import pandas as pd
from iris_repro import config, data, metrics, plotting, provenance

plt = plotting.set_style()
OUT = config.output_dir("fig1")
print("outputs ->", OUT)

## The response-gene panel\n\nModified from Han et al. 2020; see Supp. Fig. 1a.

In [ ]:
for sig in config.signals():
    genes = config.response_genes(sig)
    print(f"{config.display_name(sig):9s} ({len(genes):2d})  {', '.join(genes)}")

## Load the mouse screens\n\nBatches 1, 2 and 3 are the mESC data (Yeo et al. 2020).

In [ ]:
adata = data.load_screens()
mesc = adata[np.isin(adata.obs["batch"].values, [1, 2, 3])].copy()
print(f"{mesc.n_obs:,} cells x {mesc.n_vars:,} genes")
data.describe(mesc)

## Score the response-gene baseline

The score is the summed log-normalised expression of a pathway's response
genes — the linear rule of Supp. Fig. 1b.

In [ ]:
import scanpy as sc

norm = mesc.copy()
sc.pp.normalize_total(norm, target_sum=1e6)
sc.pp.log1p(norm)

scores = {}
for sig in config.signals():
    present = [g for g in config.response_genes(sig) if g in norm.var_names]
    missing = set(config.response_genes(sig)) - set(present)
    if missing:
        print(f"  {sig}: missing {sorted(missing)}")
    scores[sig] = metrics.response_gene_score(norm, present)
rg = pd.DataFrame(scores, index=mesc.obs_names)
rg.head()

## How well does it separate stimulated from control?

Note the threshold column: there is no shared cut-off across pathways, which
is exactly the calibration problem the paper raises.

In [ ]:
rows = []
for sig in config.signals():
    y = data.binary_labels(mesc, sig)
    rows.append({"signal": sig, **metrics.classification_metrics(y, rg[sig])})
rg_metrics = metrics.summarize_runs(rows)
rg_metrics

In [ ]:
curves = {s: {"y_true": data.binary_labels(mesc, s), "y_score": rg[s].values}
          for s in config.signals()}

fig, axes = plt.subplots(1, 2, figsize=(4.4, 2.1))
plotting.plot_roc(curves, ax=axes[0], title="Response genes: ROC")
plotting.plot_pr(curves, ax=axes[1], title="Response genes: PR")
plt.tight_layout()
plt.show()

RA, WNT and BMP reach reasonable AUROC while TGF-β and FGF lag — the pattern
reported in the text.

## Compare against IRIS

The full comparison needs the trained models. Run the script once with
`--retrain` (GPU, ~20 min) to cache predictions; afterwards it reloads them.

```bash
python figures/fig1/fig1c_insample_accuracy.py --retrain
```

In [ ]:
cache = config.output_dir("fig1") / "fig1c_iris_predictions.csv"
if cache.exists():
    iris = pd.read_csv(cache, index_col=0)
    rows = []
    for sig in config.signals():
        y = data.binary_labels(mesc, sig)
        for name, sc_ in (("IRIS", iris[sig].values), ("response gene", rg[sig].values)):
            rows.append({"signal": sig, "method": name,
                         **metrics.classification_metrics(y, sc_)})
    comparison = metrics.summarize_runs(rows)
    display(comparison.pivot(index="signal", columns="method",
                             values=["AUROC", "F1"]).round(3))
else:
    print("No cached IRIS predictions yet — run the script with --retrain.")
    print("Showing the response-gene baseline only.")
    display(rg_metrics)